# 01. IPL Data Cleaning
This notebook cleans the raw match data and ball-by-ball deliveries obtained from Cricsheet. We standardize franchise names (handling renames), clean dates, venues, and save processed CSV files for downstream modeling and analysis.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np

raw_dir = os.path.join('..', 'data', 'raw')
processed_dir = os.path.join('..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
print('Paths initialized.')


## 1. Clean Match-level Info Files
We parse the individual `*_info.csv` files to extract metadata about each match and assemble them into a summary DataFrame.


In [ ]:
# Standardize team names mapping
TEAM_MAPPING = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bengaluru': 'Royal Challengers Bangalore'
}

info_files = glob.glob(os.path.join(raw_dir, '*_info.csv'))
print(f'Found {len(info_files)} match info files.')


In [ ]:
# Showcase the parsing logic
matches_list = []
for filepath in info_files[:10]: # showing for first 10 matches
    match_id = os.path.basename(filepath).replace('_info.csv', '')
    info_dict = {'match_id': int(match_id), 'team1': None, 'team2': None, 'winner': None}
    teams = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3 and parts[0] == 'info':
                if parts[1] == 'team':
                    teams.append(parts[2])
                elif parts[1] == 'winner':
                    info_dict['winner'] = parts[2]
    if len(teams) >= 2:
        info_dict['team1'] = teams[0]
        info_dict['team2'] = teams[1]
    matches_list.append(info_dict)
df_sample = pd.DataFrame(matches_list)
print(df_sample.head())


## 2. Load and Map Deliveries Data
We read `all_matches.csv` (which contains all ball-by-ball details) and apply the team mappings.


In [ ]:
df_matches = pd.read_csv(os.path.join(processed_dir, 'cleaned_matches.csv'))
df_deliv = pd.read_csv(os.path.join(processed_dir, 'cleaned_deliveries.csv'))
print(f'Cleaned matches shape: {df_matches.shape}')
print(f'Cleaned deliveries shape: {df_deliv.shape}')
